# Clustering Some Layer

## Install TensorFlow Model Optimization Toolkit
* 설치 완료 후 반드시 Runtime 재시작!
    * '런타임' > '세션 다시 시작' 메뉴 선택'

In [1]:
!pip install tensorflow-model-optimization

## Mount Google driver

In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('g-drive mounted.')
    colab=True
except:
    print('local drive.')
    colab =False

Mounted at /content/drive
g-drive mounted.


## Import Module

In [3]:
import tensorflow as tf
import numpy as np

import tensorflow_model_optimization as tfmot
from tensorflow_model_optimization.python.core.keras.compat import keras

print(tf.__version__)
print(np.__version__)

2.19.0
1.26.4


## Load Dataset

In [4]:
(train_images, train_labels), (test_images, test_labels) = keras.datasets.mnist.load_data()

train_images = (train_images / 255.0).astype(np.float32)
test_images = (test_images / 255.0).astype(np.float32)

11490434/11490434 [==============================] - 1s 0us/step


## Load Baseline Model for MNIST (CNN)

In [5]:
model = tf.keras.models.load_model('/content/drive/MyDrive/files/save/baseline_model.h5')
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 reshape (Reshape)           (None, 28, 28, 1)         0         
                                                                 
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 16)        4624      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 16)          0         
 g2D)                                                            
                                                                 
 flatten (Flatten)           (None, 400)               0

In [6]:
_, baseline_model_accuracy = model.evaluate(
    test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)

Baseline test accuracy: 0.9900000095367432


## Baseline model weight 확인 : 추후 clustered model의 weight와 비교 예정

In [7]:
print(model.layers[3].get_weights()[0])

[[[[-0.13463008 -0.09887448 -0.13795947 ... -0.13452221 -0.03579434
     0.10540592]
   [-0.18155009  0.15832229 -0.13606851 ...  0.1204171  -0.15893538
    -0.16293684]
   [ 0.08370899 -0.05039732 -0.1176676  ... -0.02113456 -0.22348033
     0.01877562]
   ...
   [-0.0404418  -0.11982701  0.07523621 ... -0.03491493  0.06014127
    -0.45438498]
   [-0.00252522 -0.14265656 -0.06945585 ... -0.03023417  0.15696044
    -0.18880306]
   [-0.1749711   0.0695985  -0.23448451 ... -0.06049359 -0.00542932
    -0.05604602]]

  [[-0.14419478 -0.06355193 -0.19783856 ... -0.03575908  0.12289929
     0.06119277]
   [-0.35110325  0.106062   -0.37449571 ... -0.45052698 -0.1873269
     0.03811485]
   [-0.2429985  -0.00196027 -0.00464072 ... -0.05110066  0.11486182
     0.01349925]
   ...
   [-0.21528614 -0.1674511   0.14933115 ... -0.2588816  -0.3262702
    -0.05827508]
   [-0.08606555 -0.18967672 -0.00093448 ... -0.17291903 -0.35272202
    -0.23743837]
   [-0.32356033  0.1117683  -0.40611193 ...  0.0063

## Clustering Some Layer
* **Layer "conv2d_1"**
    * number of cluserts : 8
    * cluster centroids init : K-mean++
* **Layer "dense"**
    * number of cluserts : 16
    * cluster centroids init : K-mean++

In [8]:
dict_clustering_params = {
    "conv2d_1":{
        'number_of_clusters': 8,
        'cluster_centroids_init': tfmot.clustering.keras.CentroidInitialization.KMEANS_PLUS_PLUS
    },
    "dense":{
        'number_of_clusters': 16,
        'cluster_centroids_init': tfmot.clustering.keras.CentroidInitialization.KMEANS_PLUS_PLUS
    }
}

def apply_clustering_to_dense(layer:keras.layers):
  if layer.name in dict_clustering_params:
    return tfmot.clustering.keras.cluster_weights(layer, **dict_clustering_params[layer.name])
  return layer

clustered_model = keras.models.clone_model(model, clone_function = apply_clustering_to_dense)

## Compile 후 model 확인 : clustering을 위해 metadata가 추가 된 model 확인

In [9]:
clustered_model.compile(loss=keras.losses.SparseCategoricalCrossentropy(),
                        optimizer=keras.optimizers.Adam(learning_rate = 1e-5),  # 작은 leraning rate 사용
                        metrics=['accuracy'])

clustered_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 reshape (Reshape)           (None, 28, 28, 1)         0         
                                                                 
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 cluster_conv2d_1 (ClusterW  (None, 11, 11, 16)        9240      
 eights)                                                         
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 16)          0         
 g2D)                                                            
                                                        

## Clustering을 위한 training

In [10]:
hist_clustered = clustered_model.fit(
    train_images,
    train_labels,
    batch_size=500,
    epochs=1,
    validation_split=0.1
)

108/108 [==============================] - 32s 284ms/step - loss: 0.0069 - accuracy: 0.9977 - val_loss: 0.0401 - val_accuracy: 0.9910


## Accuracy 비교 : baseline model vs clustered model

In [11]:
_, clustered_model_accuracy = clustered_model.evaluate(
  test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)
print('Clustered test accuracy:', clustered_model_accuracy)

Baseline test accuracy: 0.9900000095367432
Clustered test accuracy: 0.9901000261306763


## Clustered model의 weight 확인 : 8개의 cluster로 weight가 제한 됨

In [12]:
final_model = tfmot.clustering.keras.strip_clustering(clustered_model)

print(final_model.layers[3].get_weights()[0])

[[[[-0.1238751  -0.1238751  -0.1238751  ... -0.1238751  -0.0328185
     0.11217531]
   [-0.1238751   0.11217531 -0.1238751  ...  0.11217531 -0.1238751
    -0.1238751 ]
   [ 0.11217531 -0.0328185  -0.1238751  ... -0.0328185  -0.25286335
     0.02330655]
   ...
   [-0.0328185  -0.1238751   0.11217531 ... -0.0328185   0.02330655
    -0.4580421 ]
   [ 0.02330655 -0.1238751  -0.0328185  ... -0.0328185   0.11217531
    -0.25286335]
   [-0.1238751   0.11217531 -0.25286335 ... -0.0328185  -0.0328185
    -0.0328185 ]]

  [[-0.1238751  -0.0328185  -0.25286335 ... -0.0328185   0.11217531
     0.02330655]
   [-0.25286335  0.11217531 -0.4580421  ... -0.4580421  -0.1238751
     0.02330655]
   [-0.25286335  0.02330655  0.02330655 ... -0.0328185   0.11217531
     0.02330655]
   ...
   [-0.25286335 -0.1238751   0.11217531 ... -0.25286335 -0.25286335
    -0.0328185 ]
   [-0.1238751  -0.25286335  0.02330655 ... -0.1238751  -0.25286335
    -0.25286335]
   [-0.25286335  0.11217531 -0.4580421  ...  0.023306

In [13]:
# layer 'conv_2d_1' cluster
set(final_model.layers[3].get_weights()[0].reshape(-1))

{-0.4580421,
 -0.25286335,
 -0.123875104,
 -0.032818496,
 0.023306554,
 0.11217531,
 0.23039167,
 0.28712615}

In [14]:
# layer 'dense' cluster
set(final_model.layers[6].get_weights()[0].reshape(-1))

{-0.47358844,
 -0.33315825,
 -0.26820976,
 -0.20243265,
 -0.15532047,
 -0.11495999,
 -0.077383764,
 -0.03734426,
 0.0149844345,
 0.037659016,
 0.09179689,
 0.12369282,
 0.16048053,
 0.2069147,
 0.26952597,
 0.36610103}

## 최종 clustered model의 구조 확인
* baseline model과 같음 : memory size에서도 달라진 부분이 없음 (TFMOT clustering 의 한계)

In [15]:
final_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 reshape (Reshape)           (None, 28, 28, 1)         0         
                                                                 
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 16)        4624      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 16)          0         
 g2D)                                                            
                                                                 
 flatten (Flatten)           (None, 400)               0

## TFMOT에 의해 만들어진 clustered model의 의미
* huffman coding 에 의해 압축 할 때 효율적인 압축이 가능한 형태로 변경

In [16]:
import tempfile
import os
import zipfile

def get_zipped_model_size(model):
  _, model_file = tempfile.mkstemp('.h5')
  model.save(model_file)

  _, zipped_file = tempfile.mkstemp('.zip')
  with zipfile.ZipFile(zipped_file, 'w', compression=zipfile.ZIP_DEFLATED) as f:
    f.write(model_file)

  return os.path.getsize(zipped_file)

## .zip 로 압축된 파일 크기 비교 : baseline model vs clustered model

In [17]:
size_zipped_baseline_model = get_zipped_model_size(model)
size_zipped_clustered_model = get_zipped_model_size(final_model)

print("Size of zipped baseline model file : {}".format(size_zipped_baseline_model))
print("Size of zipped clustered model file : {}".format(size_zipped_clustered_model))
print("ratio : {}".format(size_zipped_baseline_model/size_zipped_clustered_model))

/usr/local/lib/python3.12/dist-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Size of zipped baseline model file : 448158
Size of zipped clustered model file : 52020
ratio : 8.61510957324106
